#### Dependencies & Industry Standards

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# INDUSTRY CONSTANTS
TRUCK_ID = "SHL-TX-9921"
CARGO = "Low-Sulfur Diesel"
BASE_TEMP_C = 15.0      # International standard reference temperature
EXPANSION_COEFF = 0.00084 # Thermal expansion coefficient for Diesel
MAX_CAPACITY = 36000    # Liters (Standard Fuel Tanker)

# GEOSPATIAL ANCHORS
DEER_PARK_DEPOT = (29.721, -95.125)
KATY_STATION = (29.782, -95.824)

#### Defining the Operational Workflow

In [2]:
# (Phase Name, Duration in Minutes)
operational_phases = [
    ("1. Pre-Trip Inspection", 15),
    ("2. Departure to Depot", 20),
    ("3. Entry & Loading", 50),
    ("4. Stabilization (Post-Load)", 15),
    ("5. Transit to Katy", 75),
    ("6. Entry & Stabilization (Station)", 20),
    ("7. Unloading", 45),
    ("8. Return to Base", 60)
]

#### The Simulation Engine

In [3]:
def generate_telemetry():
    data = []
    current_time = datetime(2025, 4, 8, 7, 0, 0) 
    current_vol = 0
    
    # Coordinates for movement simulation
    start_lat, start_lon = DEER_PARK_DEPOT
    end_lat, end_lon = KATY_STATION
    
    for phase, duration in operational_phases:
        steps = range(0, duration, 5)
        num_steps = len(steps)
        
        for i, minute in enumerate(steps):
            # 1. Valve Logic: Open only during Loading/Unloading
            valve_status = "Open" if any(x in phase for x in ["Loading", "Unloading"]) else "Closed"
            
            # 2. Volume Logic: Fill at Depot, Drain at Station
            if "Loading" in phase:
                current_vol = MAX_CAPACITY
            elif "Unloading" in phase:
                current_vol -= (MAX_CAPACITY / num_steps)
            
            # 3. Dynamic GPS Logic: Move the truck during the "Transit" phase
            if "Transit" in phase:
                fraction = i / num_steps
                curr_lat = start_lat + (end_lat - start_lat) * fraction
                curr_lon = start_lon + (end_lon - start_lon) * fraction
            elif "Return" in phase:
                fraction = i / num_steps
                curr_lat = end_lat + (start_lat - end_lat) * fraction
                curr_lon = end_lon + (start_lon - end_lon) * fraction
            elif "Katy" in phase or "Unloading" in phase:
                curr_lat, curr_lon = KATY_STATION
            else:
                curr_lat, curr_lon = DEER_PARK_DEPOT

            # 4. Temperature Simulation: Morning rise in Houston (Celsius)
            temp = 19.0 + (len(data) * 0.08) + random.uniform(-0.2, 0.2)
            
            # 5. Physics: Observed Volume (The sensor reading)
            observed_vol = current_vol * (1 + EXPANSION_COEFF * (temp - BASE_TEMP_C))
            
            data.append({
                "timestamp": current_time,
                "phase": phase,
                "latitude": round(curr_lat, 5),
                "longitude": round(curr_lon, 5),
                "temp_c": round(temp, 2),
                "gross_volume_l": round(observed_vol, 2),
                "valve_status": valve_status,
                "vcf_applied": False 
            })
            current_time += timedelta(minutes=5)
            
    return pd.DataFrame(data)

raw_df = generate_telemetry()
print(f"Dataset generated for April 2025. Total records: {len(raw_df)}")

Dataset generated for April 2025. Total records: 60


#### Injecting the "Investigation" Anomaly

In [4]:
# 1. Identify the 'Target Phase' where theft is most likely to occur (Transit)
transit_mask = raw_df['phase'] == "5. Transit to Katy"

# 2. Define the Anomaly Volume (approx. 250 Liters)
# This represents a 'Slow Siphon' event during the drive
theft_amount = 250 

# 3. Create a gradual loss curve 
# Using a curve instead of a sudden drop makes detection harder and more realistic
loss_curve = np.linspace(0, -theft_amount, sum(transit_mask))

# 4. Apply the anomaly to the Gross Volume
raw_df.loc[transit_mask, 'gross_volume_l'] += loss_curve

# 5. Inject a 'Safety Violation' - Valve opening mid-transit
# We will simulate a sensor firing 'Open' for one 5-minute window during transit
anomaly_index = raw_df[transit_mask].index[len(raw_df[transit_mask]) // 2]
raw_df.at[anomaly_index, 'valve_status'] = "Open (ALERT)"

# 6. Save the 'Evidence' to our professional folder structure
import os
os.makedirs('data/raw', exist_ok=True)
raw_df.to_csv('data/raw/shell_katy_delivery_2025_raw.csv', index=False)

print(f"Anomaly Injected: {theft_amount}L loss added to 'Transit' phase.")
print(f"Safety Violation: Unauthorized valve state injected at index {anomaly_index}.")

Anomaly Injected: 250L loss added to 'Transit' phase.
Safety Violation: Unauthorized valve state injected at index 27.


In [9]:
raw_df.head()

,timestamp,phase,latitude,longitude,temp_c,gross_volume_l,valve_status,vcf_applied
0,2025-04-08 07:00:00,1. Pre-Trip Inspection,29.721,-95.125,18.98,0.0,Closed,False
1,2025-04-08 07:05:00,1. Pre-Trip Inspection,29.721,-95.125,19.03,0.0,Closed,False
2,2025-04-08 07:10:00,1. Pre-Trip Inspection,29.721,-95.125,19.19,0.0,Closed,False
3,2025-04-08 07:15:00,2. Departure to Depot,29.721,-95.125,19.32,0.0,Closed,False
4,2025-04-08 07:20:00,2. Departure to Depot,29.721,-95.125,19.27,0.0,Closed,False
